In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from clustering.clustering import clusterize
import json

/home/lipn/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Read data source and data target files

In [ ]:
modality_1 = pd.read_csv("modality_1.csv") #the source modality
#example: modality_1 = pd.read_csv("rna.parquet")

In [ ]:
modality_2 = pd.read_csv("modality_2.csv") #the modality to be generated
#example: modality_2 = pd.read_csv("protein.parquet")

Both datasets should have the data of the same observations(Cells, patients,...). Verify that for both modalities, that samples/observations are the index of the datasets and are aligned

In [7]:
assert (modality_1.index == modality_2.index).all(), "modality_1 and modality_2 must be aligned by index."

Here we split the data into train et test for inference

In [8]:
X_train, X_test, Y_train, Y_test = train_test_split(
        modality_1, modality_2, test_size=0.2, random_state=42
    )

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

/home/lipn/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## First step: Clustering of the source modality (Modality_1)

In [10]:
clusterize(X_train, resolution=2.0, n_neighbors=15, n_comps=50, n_pcs=30)

['ARHGEF10L' 'HIF3A' 'RNF17' ... 'GSTK1' 'SELP' 'SELS']


/home/lipn/workspace/multimodal/CrossFormer/clustering/clustering.py:39: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=resolution, key_added="leiden")


Number of clusters: 38
Clusters saved in json file 


this will create a json file in "clustering/data/clusters.json"

In [11]:
with open("clustering/data/clusters.json", "r", encoding="utf-8") as f:
    dict = json.load(f)

In [14]:
from train import train

# Second: Train CrossFormer on the train set of the source modality

In [15]:
model, x_scaler, x_imputer, y_scaler,y_imputer, cluster_names = train(
    X_train,
    Y_train,
    dict,
    device=device,
    d_model=128,
    n_heads=8,
    n_layers=2,
    n_queries=32,
    z_dim=64,
    tok_hidden=256,
    dropout=0.3,
    lr=3e-4,
    weight_decay=1e-4,
    batch_size=32,
    epochs=150
)

Using pathways: 38
Epoch 001 | loss 0.3892
Epoch 002 | loss 0.3799
Epoch 003 | loss 0.3661
Epoch 004 | loss 0.3514
Epoch 005 | loss 0.3387
Epoch 006 | loss 0.3297
Epoch 007 | loss 0.3219
Epoch 008 | loss 0.3154
Epoch 009 | loss 0.3076
Epoch 010 | loss 0.3011
Epoch 011 | loss 0.2947
Epoch 012 | loss 0.2885
Epoch 013 | loss 0.2841
Epoch 014 | loss 0.2808
Epoch 015 | loss 0.2768
Epoch 016 | loss 0.2728
Epoch 017 | loss 0.2701
Epoch 018 | loss 0.2679
Epoch 019 | loss 0.2661
Epoch 020 | loss 0.2623
Epoch 021 | loss 0.2620
Epoch 022 | loss 0.2601
Epoch 023 | loss 0.2577
Epoch 024 | loss 0.2554
Epoch 025 | loss 0.2524
Epoch 026 | loss 0.2507
Epoch 027 | loss 0.2492
Epoch 028 | loss 0.2475
Epoch 029 | loss 0.2450
Epoch 030 | loss 0.2460
Epoch 031 | loss 0.2440
Epoch 032 | loss 0.2443
Epoch 033 | loss 0.2414
Epoch 034 | loss 0.2401
Epoch 035 | loss 0.2397
Epoch 036 | loss 0.2398
Epoch 037 | loss 0.2368
Epoch 038 | loss 0.2368
Epoch 039 | loss 0.2337
Epoch 040 | loss 0.2327
Epoch 041 | loss 0.23

# Finally, generad the target modality

In [16]:
from inference import infer_and_generate

In [18]:
generated_modality = infer_and_generate(
    model=model,
    x_imputer=x_imputer,
    x_scaler=x_scaler,
    y_scaler=y_scaler,
    modality_1_test=X_test,
    target_columns=Y_test.columns,
    device=device
)

In [19]:
generated_modality.shape

(186, 189)

In [18]:
generated_modality.to_csv("generated_modality.csv")